In [1]:
# task 4.1

# create database
import sqlite3
db = sqlite3.connect('Booking.db')

# dropping tables to avoid duplicates, for debugging
db.execute('''DROP TABLE IF EXISTS Villa''')
db.execute('''DROP TABLE IF EXISTS CustomerBooking''')

# create Villa table
db.execute('''CREATE TABLE Villa(
    VillaID INTEGER,
    VillaName TEXT,
    Country TEXT,
    Cost INTEGER,
    PRIMARY KEY (VillaID)
    )''')

# create CustomerBooking table
db.execute('''CREATE TABLE CustomerBooking(
    BookingID INTEGER,
    CustomerID INTEGER,
    VillaID INTEGER,
    StartDate TEXT,
    NoOfDays INTEGER,
    PRIMARY KEY (BookingID),
    FOREIGN KEY (VillaID) REFERENCES Villa(VillaID)
    )''')

db.close()

In [2]:
# task 4.2

# function to read from textfiles and return data as a 2D list
def read_data(textfile):
    data = [] # list of info
    with open(textfile,'r') as file: # file closes automatically
        for line in file:
            line = line.strip() # remove line break
            line = line.split(',') # split data by comma
            
            data.append(line)
    return data 

# connect to database
db = sqlite3.connect('Booking.db')

# first table: villa
villas = read_data('villas.txt')
for villa in villas:    
    db.execute('''INSERT INTO Villa(VillaID, VillaName, Country, Cost) \
    VALUES (?,?,?,?)''', tuple(villa)) # change datatype to a tuple
    db.commit()

# second table: customer booking
bookings = read_data('customerBookings.txt')
for booking in bookings:
    db.execute('''INSERT INTO CustomerBooking(BookingID, CustomerID, VillaID, StartDate, NoOfDays) \
    VALUES(?,?,?,?,?)''',tuple(booking)) # change datatype to a tuple
    db.commit()

    
# testing
print('Information in Villa table')
cursor = db.execute('SELECT * FROM Villa')
results = cursor.fetchall()
for row in results:
    print(row)
print()

print('Information in Customer Booking table')
cursor = db.execute('SELECT * FROM CustomerBooking')
results = cursor.fetchall()
for row in results:
    print(row)
print()

db.close()

Information in Villa table
(1, 'Rose', 'France', 128)
(2, 'Sea view', 'Australia', 325)
(3, 'Dolphin', 'New Zealand', 490)
(4, 'Flower haven', 'Mexico', 580)
(5, 'Mountain breeze', 'India', 268)
(6, 'Sunset', 'UK', 136)
(7, 'Moonlight', 'USA', 358)
(8, 'White brick', 'Italy', 410)
(9, 'Blue house', 'Germany', 400)
(10, 'Walled garden', 'Croatia', 258)

Information in Customer Booking table
(1, 857, 3, '05-Jan', 7)
(2, 1149, 10, '20-Jan', 11)
(3, 388, 9, '05-Feb', 2)
(4, 230, 4, '03-Mar', 14)
(5, 1254, 6, '19-Nov', 10)
(6, 1500, 4, '12-May', 4)
(7, 1687, 7, '03-Mar', 6)
(8, 1408, 2, '18-Aug', 2)
(9, 1138, 3, '02-Mar', 4)
(10, 235, 5, '04-Apr', 7)
(11, 4, 10, '07-Jul', 12)
(12, 1981, 8, '15-Jan', 5)
(13, 500, 9, '06-Nov', 4)
(14, 332, 9, '03-Apr', 7)
(15, 1512, 5, '12-Oct', 14)
(16, 660, 4, '08-Jul', 6)
(17, 738, 4, '09-Aug', 7)
(18, 1211, 1, '12-Nov', 3)
(19, 1089, 1, '15-Sep', 4)
(20, 1294, 2, '03-Feb', 14)
(21, 1087, 3, '10-Apr', 14)
(22, 722, 3, '01-Feb', 2)
(23, 566, 6, '13-Apr', 21

In [3]:
# task 4.3 
# Method 1: using datetime module

# import datetime to add dates
import datetime

# write a function to generate dates
def generate_dates(StartDate, NoOfDays):
    # matching month to integer
    MonthToInt = {'Jan': 1, 'Feb' : 2, 'Mar': 3, 'Apr': 4,
                'May': 5, 'Jun': 6, 'Jul': 7, 'Aug': 8,
                'Sep': 9, 'Oct': 10, 'Nov': 11, 'Dec': 12} 
    
    # locating month by index
    Months = ['', 'Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
    
    list_dates = []
    date = StartDate
    for i in range(NoOfDays):
        list_dates.append(date)
        
        # extracting the day and month of current date as integers
        day = int(date[:2])
        month = MonthToInt[date[3:]]

        # update to the next date
        full_date = datetime.date(2025, month, day) + datetime.timedelta(days = 1)

        # update day in text, add 0 if necessary
        if full_date.day < 10 : # OR len(next_full_date.day) == 1:
            date = '0' + str(full_date.day) + '-'
        else:
            date = str(full_date.day) + '-'
            
        # update month in text
        date += Months[full_date.month]
     
    return list_dates

# connect to database
db = sqlite3.connect('Booking.db')

# # drop table to ensure no dupplicates, for debugging
db.execute('DROP TABLE IF EXISTS Villa_Booking')

# create Villa_Booking table
db.execute('''CREATE TABLE Villa_Booking(
    VillaID INTEGER,
    DateBooked TEXT,
    PRIMARY KEY (VillaID, DateBooked)
    FOREIGN KEY('VillaID') REFERENCES 'Villa'('VillaID')
    )''') # both villaID and DateBooked form a composite key to identify each row 


# extract information from CustomerBooking table
cursor = db.execute('''SELECT VillaID, StartDate, NoOfDays FROM CustomerBooking''')
results = cursor.fetchall()
for result in results:
    villaID, StartDate, NoOfDays = result[0],result[1],result[2] 
    
    days = generate_dates(StartDate, NoOfDays)
    for date in days:
        db.execute('INSERT INTO Villa_Booking(VillaID,DateBooked) VALUES(?,?)',(villaID, date))
        
        
db.commit()        
db.close()

In [4]:
# task 4.3 
# Method 2: without datetime module

# write a function to generate dates
def generate_dates(StartDate, NoOfDays):
    # matching months to the total number of days within the month
    # we may use the calendar at the bottom right corner of the laptop
    MonthTotal = {'Jan': 31, 'Feb' : 28, 'Mar': 31, 'Apr': 30,
                'May': 31, 'Jun': 30, 'Jul': 31, 'Aug': 31,
                'Sep': 30, 'Oct': 31, 'Nov': 30, 'Dec': 31} 
    
    # locating month text by index
    Months = ['', 'Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
    
    list_dates = []
    date = StartDate
    for i in range(NoOfDays):
        list_dates.append(date)
        # extracting the day and month of current date as integers
        day = int(date[:2]) + 1
        month = date[3:]
        if day > MonthTotal[month]:
            date = '01-' + Months[Months.index(month) + 1]
        elif day < 10:
            date = '0' + str(day) + '-' + month
        else:
            date = str(day) + '-' + month
    
    return list_dates
                         

# connect to database
db = sqlite3.connect('Booking.db')

# # drop table to ensure no dupplicates, for debugging
db.execute('DROP TABLE IF EXISTS Villa_Booking')

# create Villa_Booking table
db.execute('''CREATE TABLE Villa_Booking(
    VillaID INTEGER,
    DateBooked TEXT,
    PRIMARY KEY (VillaID, DateBooked)
    FOREIGN KEY('VillaID') REFERENCES 'Villa'('VillaID')
    )''') # both villaID and DateBooked form a composite key to identify each row 


# extract information from CustomerBooking table
cursor = db.execute('''SELECT VillaID, StartDate, NoOfDays FROM CustomerBooking''')
results = cursor.fetchall()
for result in results:
    villaID, StartDate, NoOfDays = result[0],result[1],result[2] 
    
    days = generate_dates(StartDate, NoOfDays)
    for date in days:
        db.execute('INSERT INTO Villa_Booking(VillaID,DateBooked) VALUES(?,?)',(villaID, date))
        
        
db.commit()        
db.close()

In [5]:
# task 4.4

# take inputs from user
villaname = input('Enter villa name:')
month = input('Enter month (short form):')
date = input('Enter date:')
NoOfDays = int(input('Enter number of days:'))

# find villaID through villa name
db = sqlite3.connect('Booking.db')
cursor = db.execute('SELECT VillaID FROM Villa WHERE VillaName = ?', (villaname, ))
villaID = cursor.fetchone()[0]

# format starting date
if len(date) == 1:
    date = '0' + date
StartDate = date +'-'+ month

# generate the dates to be checked
days = generate_dates(StartDate, NoOfDays)

# list of available and unavailable dates
avail_dates = []
unavail_dates = [] 

for date in days:
    # search for bookings on the date for the villa
    cursor = db.execute('''SELECT * FROM Villa_Booking WHERE VillaID = ? AND DateBooked = ?''', (villaID, date))
    result = cursor.fetchone()
    
    if result == None: # date is available
        avail_dates.append(date)
    else: 
        unavail_dates.append(date)
        
# output
print('List of dates available for booking: {}'.format(','.join(avail_dates)))
print('List of dates not available for booking: {}'.format(','.join(unavail_dates)))

cursor.close()
db.close()

Enter villa name:Dolphin
Enter month (short form):Apr
Enter date:8
Enter number of days:4
List of dates available for booking: 08-Apr,09-Apr
List of dates not available for booking: 10-Apr,11-Apr
